In [2]:
import numpy as np
import torch
import os
import sys
import h5py
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.insert(0, parent_dir)
from model import AttentionModel
from model_PCA_correlation import AttentionModel_PCA
from dcascore import *
from utils import read_fasta_alignment, remove_duplicate_sequences, add_PCA_coords, quickread

# back to original path (in PLM)
sys.path.pop(0)  # Removes the parent_dir from sys.path
from model import AttentionModel
from model_PCA_correlation import AttentionModel_PCA_once

from plm_gen_methods import generate_plm_n_save, generate_coords_n_save, generate_multiple_targets_n_save,generate_plm_vect_n_save
from seq_utils import read_tensor_from_txt, set_seed, letters_to_nums, modify_seq 


alessandro vs mod code no PCA

In [22]:
def compute_product_Q_K(Q, K):
    
    

    H, _, N = Q.shape
    # Step 1: Compute the raw attention scores using einsum
    e = torch.einsum('hdi,hdj->ijh', Q, K)  # Shape: (N, N, H)

    # Exclude self-interactions by setting scores to -inf on the diagonal
    # i_indices = torch.arange(N).unsqueeze(1)
    # j_indices = torch.arange(N).unsqueeze(0)
    # self_mask = (i_indices != j_indices).float()
    # mask_value = -1e9  # A large negative value to zero out after softmax
    # e = e * self_mask.unsqueeze(-1) + (1 - self_mask.unsqueeze(-1)) * mask_value

    # If there's a domain split:
    # if self.index_last_domain1 > 0 and self.index_last_domain1 < N:
    #     domain_masks = self.create_attention_masks(
    #         H=H, 
    #         L=N, 
    #         index_last_domain1=self.index_last_domain1,
    #         H1=self.H1,  # per your original usage
    #         H2=self.H2
    #     )
    #     # Invert the domain masks to identify positions to mask
    #     inverted_domain_masks = (1 - domain_masks).bool()  # Positions to mask are True
    #     # Permute e to match the shape of domain_masks
    #     e = e.permute(2, 0, 1)  # Shape: (H, N, N)
    #     # Apply the masks
    #     e = e.masked_fill(inverted_domain_masks, mask_value)
    #     # Permute e back to original shape
    #     e = e.permute(1, 2, 0)  # Shape: (N, N, H)
    # else:
    #     domain_masks = 0

    return e

def compute_attention_heads( Q, K, V, index_last_domain1=0, H1=0, H2=0):
    
    

    H, _, N = Q.shape
    # N, _, _ = V.shape  # Actually your code re-assigns same N but let's keep it as is
    _N, _, _ = V.shape
    index_first_domain2 = index_last_domain1 + 1

    # Get e from compute_product_Q_K
    e = compute_product_Q_K(Q, K)

    sf = torch.zeros(N, N, H)
    for h in range(H):
        if index_last_domain1 != 0:
            if h < H1:
                # Heads for Domain 1
                softmax_vals = torch.softmax(
                    e[0:index_last_domain1+1, 0:index_last_domain1+1, h], 
                    dim=1
                )
                top = torch.cat([
                    softmax_vals,
                    torch.zeros(
                        index_last_domain1+1, 
                        N - index_last_domain1 - 1, 
                        device=device
                    )
                ], dim=1)
                sf_domain = torch.cat([
                    top,
                    torch.zeros(
                        N - index_last_domain1 - 1, 
                        N, 
                        device=device
                    )
                ], dim=0)
                sf = sf.clone()
                sf[:, :, h] = sf_domain
            elif h < H2:
                # Heads for Domain 2
                softmax_vals = torch.softmax(
                    e[index_first_domain2:, index_first_domain2:, h], 
                    dim=1
                )
                # Create the top-left zero block
                bottom_left = torch.zeros(
                    N - index_first_domain2, 
                    index_first_domain2, 
                    device=device
                )
                # top and bottom
                top = torch.zeros(index_first_domain2, N, device=device)
                bottom = torch.cat([bottom_left, softmax_vals], dim=1)
                sf_domain = torch.cat([top, bottom], dim=0)
                sf = sf.clone()
                sf[:, :, h] = sf_domain
            else:
                # Heads for inter-domain interactions
                sf_domain = torch.softmax(e[:, :, h], dim=1)
                sf = sf.clone()
                sf[:, :, h] = sf_domain
        else:
            # No domain masks applied
            sf_domain = torch.softmax(e[:, :, h], dim=1)
            print("sf_domain" , sf_domain.shape)
            sf = sf.clone()
            sf[:, :, h] = sf_domain

    return sf

def compute_mat_ene( Q, K, V, Z, H1=0, H2=0, index_last_domain1=0):
    
    # We'll assume you intended to call self.compute_attention_heads here.
    # The old snippet references 'e' out of nowhere, so presumably that was from compute_product_Q_K.
    # We'll keep the lines exactly the same, except we clarify how 'sf' is obtained.

    # The code below uses variables that appear in the snippet,
    # but to keep it consistent, let's define them in place:
    

    H, q, _ = V.shape
    # For clarity in your snippet, 'e' was from compute_product_Q_K,
    # and 'sf' is from compute_attention_heads.
    # We'll compute them inside to match your logic:

    sf =compute_attention_heads(
        Q=Q, 
        K=K, 
        V=V, 
        index_last_domain1=index_last_domain1, 
        H1=H1, 
        H2=H2
    )

    # From your snippet, you used e.shape for N_e1, N_e2, H_e,
    # but actually let's just read from sf itself.
    N_e1, N_e2, H_e = sf.shape
    N_Z, M = Z.shape

    assert N_e1 == N_e2 == N_Z, "Mismatch in N between sf and Z"
    N = N_e1
    #new testing
    i_indices = torch.arange(N).unsqueeze(1)
    j_indices = torch.arange(N).unsqueeze(0)
    self_mask = (i_indices != j_indices).float().to(Q.device)

    sf = sf * self_mask.unsqueeze(-1)
    mat_ene = torch.zeros(N, q, M)

    # Weighted sum loop
    for h in range(H):
        V_h = V[h]
        # The next line in your snippet references V_h[:, Z], 
        # but that can be tricky because Z is shape (N, M).
        # We keep it as it is in your snippet, trusting you have reason:
        V_h_Zj = V_h[:, Z]     # shape => (q, N, M)
        V_h_Zj = V_h_Zj.permute(1, 0, 2)  # => (N, q, M)

        mat_ene_h = torch.einsum('ij,jqm->iqm', sf[:, :, h], V_h_Zj)
        mat_ene += mat_ene_h

    mat_ene = mat_ene.permute(1, 0, 2)
    return mat_ene, sf

def loss_wo_J( Q, K, V, Z, weights, lambd=0.001, index_last_domain1=0, H1=0, H2=0):
    
    

    H, d, N = Q.shape
    q = V.shape[1]  # Number of amino acids
    M = Z.shape[1]

    # Step: compute mat_ene and sf
    mat_ene, sf = compute_mat_ene(
        Q, 
        K, 
        V, 
        Z, 
        H1=H1, 
        H2=H2, 
        index_last_domain1=index_last_domain1
    )  # Shape: (q, N, M)

    # logsumexp
    lge = torch.logsumexp(mat_ene, dim=0)  # Shape: (N, M)
    
    Z_indices = Z.unsqueeze(0)  # Shape: (1, N, M)
    mat_ene_selected = torch.gather(mat_ene, dim=0, index=Z_indices).squeeze(0)  # (N, M)

    pl_elements = weights * (mat_ene_selected - lge) #weighted sum along M
    pl = -torch.sum(pl_elements) # sum along N
    loss_j= -torch.sum(mat_ene_selected)
    loss_log = - torch.sum(-lge)
    print("ale loss j : ", loss_j.item())
    print("ale loss log: ", loss_log.item())
    # For the regularization term, your snippet references M_matrix, etc.
    # That part of your snippet uses `self_mask`, but it was never fully spelled out. 
    # We'll keep it exactly as in your snippet:

    # The snippet tries: M_matrix = torch.einsum('ijh,ijk,ij->hk', sf, sf, self_mask)
    # But 'self_mask' is not defined in this scope. If that was part of your code, 
    # you must define it. We'll keep the line as is (though it may error if `self_mask` is missing).
# Compute regularization term
    i_indices = torch.arange(N).unsqueeze(1)
    j_indices = torch.arange(N).unsqueeze(0)
    self_mask = (i_indices != j_indices).float()
    M_matrix = torch.einsum('ijh,ijk,ij->hk', sf, sf, self_mask)  # Shape: (H, H)
    print("V shape",V.shape)
    print(H)
    VV = V.reshape(H, -1)  # Shape: (H, q*q)
    VV_T = VV @ VV.T  # Shape: (H, H)
    sum_J_squared = torch.sum(M_matrix * VV_T)  # Scalar
    reg = lambd * sum_J_squared  # Scalar
    print("ale loss: ", pl.item())
    print("ale reg: ",reg.item())
    loss_value = pl + reg

    del sf, mat_ene, mat_ene_selected, M, VV_T
    torch.cuda.empty_cache()

    return loss_value.item()

def loss_by_hand(J,Z,weights,lambd=0.001):
    loss_1=0
    loss_2=0
    L=J.shape[-1]
    q=J.shape[0]
    for seq in Z:
        for i in range(L):
            for j in range(L):
                if i==j:
                    continue
                loss_1+=J[seq[i],seq[j],i,j]
            val=0
            for a in range(q):
                inside_exp=0
                for j in range(L):
                    if i==j:
                        continue
                    inside_exp+=J[a,seq[j],i,j]
                val+=np.exp(inside_exp)
            loss_2-=np.log(val)
    print("by hand loss_:",-loss_1.item())
    print("by hand loss log:",-loss_2.item())
    loss=loss_1+loss_2
    reg= lambd*np.einsum("ijab,ijab",J,J)
    print("by hand reg:",reg)
    loss_tot=-loss+reg
    return loss_tot.item()


                

In [11]:
H = 64
d= 10
N = 174
n_epochs = 500
nb_PCA_comp=2
loss_type = 'without_J'
family = 'jdoms' #'jdoms_bacteria_train2'
cwd = parent_dir
Q_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_youss/Q_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
K_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_youss/K_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
V_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_youss/V_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
H,d,N=Q_1.shape
q=V_1.shape[1]
model=AttentionModel(H,d,N,q,Q=Q_1,V=V_1,K=K_1)
torch.sum(model.Q-Q_1)
device = Q_1.device
L = Q_1.shape[-1]
W=attention_heads_from_model(model,Q_1,K_1,V_1)
print(W.shape)

i_indices = torch.arange(L, device=device).unsqueeze(1)
j_indices = torch.arange(L, device=device).unsqueeze(0)
mask = (i_indices != j_indices).float().unsqueeze(0)  # shape (1, L, L)
W = W * mask
    
# Compute Jtens
Jtens = torch.einsum('hri,hab->abri', W, V_1)  # Shape: (q, q, L, L)
q = Jtens.shape[0]
N = Jtens.shape[2]
print(q)
print(N)
print(Jtens.shape)
print(Jtens.shape[-1])
print(Q_1.dtype)

torch.Size([64, 63, 63])
21
63
torch.Size([21, 21, 63, 63])
63
torch.float32


In [14]:

def compute_mat_ene_cross_loop(Q, K, V, Z1, Z2, H1=0, H2=0, index_last_domain1=0):
    

    H, q1, q2 = V.shape
    N1, M = Z1.shape
    N2 = Z2.shape[0]

    sf = compute_attention_heads(Q, K, V, H1=H1, H2=H2, index_last_domain1=index_last_domain1)

    mat_ene = torch.zeros(q1, N1, M, device=device)

    for h in range(H):
        V_h = V[h]  # (q1, q2)

        # Expand indices
        Z1_exp = Z1[:, None, :].expand(N1, N2, M)  # (N1, N2, M)
        Z2_exp = Z2[None, :, :].expand(N1, N2, M)  # (N1, N2, M)

        # Flatten and index into V_h
        Z2_flat = Z2_exp.reshape(-1)
        V_selected_flat = V_h[:, Z2_flat]  # (q1, N1*N2*M)

        V_selected = V_selected_flat.view(q1, N1, N2, M)  # (q1, N1, N2, M)

        sf_h = sf[:, :, h]  # (N1, N2)

        mat_ene_h = torch.einsum('ij,qijm->qim', sf_h, V_selected)
        mat_ene += mat_ene_h

    return mat_ene, sf

# ----------------- Vectorized implementation -----------------
def compute_mat_ene_cross_vec(Q, K, V, Z1, Z2, H1=0, H2=0, index_last_domain1=0):
    

    H, q1, q2 = V.shape
    N1, M = Z1.shape
    N2 = Z2.shape[0]

    sf = compute_attention_heads(Q, K, V, H1=H1, H2=H2, index_last_domain1=index_last_domain1)

    # Expand indices
    Z2_exp = Z2[None, :, :].expand(N1, N2, M)  # (N1, N2, M)

    # Gather V across all heads at once
    V_Z2 = V[:, :, Z2_exp]             # (H, q1, N2, M)
    V_Z2 = V_Z2.permute(1, 2, 3, 0)    # (q1, N2, M, H)
    V_Z2 = V_Z2.unsqueeze(0).expand(N1, -1, -1, -1, -1)  # (N1, q1, N2, M, H)

    # sf: (N1, N2, H) -> add q1 and M dims for broadcast
    sf_exp = sf.unsqueeze(1).unsqueeze(3)  # (N1, 1, N2, 1, H)

    # Weighted sum over N2 and H
    mat_ene = torch.einsum('nqh, nqmhq -> qnm', sf, V_Z2)  # (q1, N1, M)

    return mat_ene, sf
from seq_utils import letters_to_nums, sequences_from_fasta, one_hot_seq_batch
family = 'jdoms_bacteria_train2'
file_test_data=r"C:\Users\youss\OneDrive\Bureau\master epfl\MA2\TP4 De los Rios\git_test\PLM-gen-DCA\Attention-DCA-main\CODE\DataAttentionDCA\jdoms\jdoms_bacteria_train2.fasta"

train_sequences = sequences_from_fasta(file_test_data)
train_sequences_num = [letters_to_nums(seq) for seq in train_sequences]
z_test=np.array(train_sequences_num[:1])
print(z_test.shape)
weights=np.array([1])
print(weights.shape)
Z, W = quickread(file_test_data,max_gap_frac=0.8)
Z=add_PCA_coords(Z.T,35).T
Z1=torch.from_numpy(Z[:-2,:])
Z2=torch.from_numpy(Z[-2:,:])
out_loop, sf_loop = compute_mat_ene_cross_loop(Q_1, K_1, V_1, Z1,Z2)
out_vec, sf_vec = compute_mat_ene_cross_vec(Q_1, K_1, V_1, Z1,Z2)


print("Max difference between loop and vectorized:", (out_loop - out_vec).abs().mean().item())



(1, 63)
(1,)
Total sequences read: 14502
Sequences after filtering: 14502
Sampling 100000 pairs out of 105146751 total pairs.
Mean fraction of identical positions (sampled): 0.3708750145371837
Computed theta: 0.3278732598143477


100%|██████████| 14502/14502 [00:16<00:00, 889.25it/s] 


3265.70225762244


IndexError: index 25 is out of bounds for dimension 0 with size 21

In [9]:
print("Max difference between loop and vectorized:", (out_loop ).abs().min().item())
print("Max difference between loop and vectorized:", (out_loop).abs().mean().item())

Max difference between loop and vectorized: 1.9371509552001953e-07
Max difference between loop and vectorized: 1.2998921871185303


In [3]:
from scipy.special import softmax
set_seed()
H = 64
d= 10
N = 174
q=21
n_epochs = 500
nb_PCA_comp=2
L=63
loss_type = 'without_J'
family = 'jdoms' #'jdoms_bacteria_train2'
import h5py

f = h5py.File("matrices.jld2", "r")
Q_1 = f["Q"][:]
K_1 = f["K"][:]
V_1 = f["V"][:]
J = f["J"][:]
cwd = parent_dir
# Q_1 = np.loadtxt( cwd +"/results/Pagnani_julia_results/Q_tensor.txt")
# K_1 = np.loadtxt( cwd +"/results/Pagnani_julia_results/K_tensor.txt")
# V_1 = np.loadtxt( cwd +"/results/Pagnani_julia_results/V_tensor.txt")
print(Q_1.shape)
Q_1=torch.from_numpy(Q_1)
Q_1=Q_1.permute(2,1,0)
print(Q_1.shape)
K_1=torch.from_numpy(K_1)
K_1=K_1.permute(2,1,0)
print(K_1.shape)
V_1=torch.from_numpy(V_1)
V_1=V_1.permute(2,1,0)
print(V_1.shape)
J=torch.from_numpy(J)
print(J.shape)
Q_1=Q_1.float()
K_1=K_1.float()
V_1=V_1.float()
model=AttentionModel(H,d,N,q,Q=Q_1,V=V_1,K=K_1)
torch.sum(model.Q-Q_1)
device = Q_1.device
L = Q_1.shape[-1]
W=attention_heads_from_model(model,Q_1,K_1,V_1)
print(W.shape)
J=J.permute(1,0,3,2)
i_indices = torch.arange(L, device=device).unsqueeze(1)
j_indices = torch.arange(L, device=device).unsqueeze(0)
mask = (i_indices != j_indices).float().unsqueeze(0)  # shape (1, L, L)
W = W * mask
W = W.float()
V_1 = V_1.float()

# Compute Jtens
sf=np.zeros((H,L,L))
perso_mask=np.ones((L,L))-np.eye(L)
# for h in range(H):
#     sf[h,:,:]=torch.softmax(torch.from_numpy(np.einsum("di,dj->ij",Q_1[h,:,:],K_1[h,:,:])),axis=1)

# J_tens_by_hand=np.einsum("hab,hij,ij->abij",V_1,sf,perso_mask)
Jtens = torch.einsum('hri,hab->abri', W, V_1)  # Shape: (q, q, L, L)
print(Jtens.shape)
diff=(J-Jtens)**2
var=J**2
print("difference between tensors",diff.mean().item())
print("var one J",var.mean().item())
sf_masked=torch.from_numpy(sf)*mask


(63, 10, 64)
torch.Size([64, 10, 63])
torch.Size([64, 10, 63])
torch.Size([64, 21, 21])
torch.Size([21, 21, 63, 63])
torch.Size([64, 63, 63])
torch.Size([21, 21, 63, 63])
difference between tensors 5.347133284794639e-16
var one J 0.003579701802894185


verification loss avec deux tenseurs

In [23]:
def compute_product_Q_K( Q, K):
    
    

    H, _, N = Q.shape
    # Step 1: Compute the raw attention scores using einsum
    e = torch.einsum('hdi,hdj->ijh', Q, K)  # Shape: (N1, N2, H)

    

    return e

def compute_attention_heads( Q, K, V, index_last_domain1=0, H1=0, H2=0):
    
    
    index_first_domain2 = index_last_domain1 + 1

    # Get e from compute_product_Q_K
    e = compute_product_Q_K(Q, K)
    N1,N2,H=e.shape

    sf = torch.zeros(N1, N2, H, device=device)
    for h in range(H):
        if index_last_domain1 != 0:
            pass #Again no masks 
            
        else:
            # No domain masks applied
            sf_domain = torch.softmax(e[:, :, h], dim=1)
            sf = sf.clone()
            sf[:, :, h] = sf_domain

    return sf #shape (N1,N2,H)

def compute_mat_ene( Q, K, V, Z, H1=0, H2=0, index_last_domain1=0):
    
    # We'll assume you intended to call self.compute_attention_heads here.
    # The old snippet references 'e' out of nowhere, so presumably that was from compute_product_Q_K.
    # We'll keep the lines exactly the same, except we clarify how 'sf' is obtained.

    # The code below uses variables that appear in the snippet,
    # but to keep it consistent, let's define them in place:
   
    H, q, _ = V.shape
    # For clarity in your snippet, 'e' was from compute_product_Q_K,
    # and 'sf' is from compute_attention_heads.
    # We'll compute them inside to match your logic:

    sf = compute_attention_heads(
        Q=Q, 
        K=K, 
        V=V, 
        index_last_domain1=index_last_domain1, 
        H1=H1, 
        H2=H2
    )

    # From your snippet, you used e.shape for N_e1, N_e2, H_e,
    # but actually let's just read from sf itself.
    N_e1, N_e2, H_e = sf.shape
    N_Z, M = Z.shape

    assert N_e1 == N_e2 == N_Z, "Mismatch in N between sf and Z"
    N = N_e1
    i_indices = torch.arange(N).unsqueeze(1)
    j_indices = torch.arange(N).unsqueeze(0)
    self_mask = (i_indices != j_indices).float().to(Q.device)

    sf = sf * self_mask.unsqueeze(-1)
    mat_ene = torch.zeros(N, q, M, device=device)

    # Weighted sum loop
    for h in range(H):
        V_h = V[h]
        # The next line in your snippet references V_h[:, Z], 
        # but that can be tricky because Z is shape (N, M).
        # We keep it as it is in your snippet, trusting you have reason:
        V_h_Zj = V_h[:, Z]     # shape => (q, N, M)
        V_h_Zj = V_h_Zj.permute(1, 0, 2)  # => (N, q, M)

        mat_ene_h = torch.einsum('ij,jqm->iqm', sf[:, :, h], V_h_Zj)
        mat_ene += mat_ene_h

    mat_ene = mat_ene.permute(1, 0, 2)
    return mat_ene, sf
def compute_mat_ene_cross( Q, K, V, Z1, Z2, H1=0, H2=0, index_last_domain1=0):
    """
    Q: Tensor (H, d, N1)
    K: Tensor (H, d, N2)
    V: Tensor (H, q1, q2)
    Z1: LongTensor (N1, M)
    Z2: LongTensor (N2, M)
    sf: attention scores (computed from Q, K): (N1, N2, H)
    
    Returns:
        mat_ene: Tensor (q1, N1, M) — same structure as compute_mat_ene
        sf: Tensor (N1, N2, H) — attention weights
    """
   

    H, q1, q2 = V.shape
    N1, M = Z1.shape
    N2 = Z2.shape[0]

    sf = compute_attention_heads(
        Q=Q, K=K, V=V, H1=H1, H2=H2, index_last_domain1=index_last_domain1
    )  # shape: (N1, N2, H)

    # Final energy: (q1, N1, M)  (same as compute_mat_ene)
    mat_ene = torch.zeros(q1, N1, M, device=device)

    for h in range(H):
        V_h = V[h]  # (q1, q2)

        # Z1: (N1, M) → (N1, 1, M)
        Z1_exp = Z1[:, None, :].expand(N1, N2, M)  # (N1, N2, M)
        Z2_exp = Z2[None, :, :].expand(N1, N2, M)  # (N1, N2, M)

        # Flatten to (N1*N2*M,)
        Z1_flat = Z1_exp.reshape(-1)
        Z2_flat = Z2_exp.reshape(-1)

        # Index into V_h[q1, q2] → (N1*N2*M, q1)
        V_selected_flat = V_h[:, Z2_flat]  # (q1, N1*N2*M)

        # Reshape to (q1, N1, N2, M)
        V_selected = V_selected_flat.view(q1, N1, N2, M)

        sf_h = sf[:, :, h]  # (N1, N2)

        # Weighted sum over j (dim=2, the N2 dimension)
        mat_ene_h = torch.einsum('ij,qijm->qim', sf_h, V_selected)  # (q1, N1, M)

        mat_ene += mat_ene_h

    return mat_ene, sf  # mat_ene: (q1, N1, M)



def compute_loss(QJ, KJ, VJ, Z, 
                    QG, KG, VG, Z1, Z2, weights,lambd=0.001,
                    H1=0, H2=0, index_last_domain1=0):
    """
    Computes the loss L(J) from the given formula.

    Args:
        QJ, KJ, VJ: tensors for J computation (self energy)
        Z: LongTensor (N1, M), indices for J
        QG, KG, VG: tensors for G computation (cross energy)
        Z1: LongTensor (N1, M), indices for sequence 1
        Z2: LongTensor (N2, M), indices for sequence 2
        H1, H2, index_last_domain1: attention settings

    Returns:
        loss: scalar tensor
    """

    # ------------------------
    # Step 1: Compute J (self-energy)
    # ------------------------
    # mat_ene_J: (q1, N1, M)+regularization
    mat_ene_J, sf_J = compute_mat_ene(
        Q=QJ, K=KJ, V=VJ, Z=Z,
        H1=H1, H2=H2, index_last_domain1=index_last_domain1
    )
    
    H, d, N1 = QJ.shape
    M_matrix = torch.einsum('ijh,ijk->hk', sf_J, sf_J)  # (H, H)
    VV = VJ.view(H, -1)  # (H, q1*q2)
    VV_T = VV @ VV.T  # (H, H)
    sum_J_squared = torch.sum(M_matrix * VV_T)  # scalar
    reg1 = lambd * sum_J_squared
    # ------------------------
    # Step 2: Compute G (cross-energy)
    # ------------------------
    # mat_ene_G: (q1, N1, M)+regularization
    mat_ene_G, sf_G = compute_mat_ene_cross(
        Q=QG, K=KG, V=VG, Z1=Z1, Z2=Z2,
        H1=H1, H2=H2, index_last_domain1=index_last_domain1
    )
    H, d, N1 = QG.shape
    M_matrix = torch.einsum('ijh,ijk->hk', sf_G, sf_G)  # (H, H)
    VV = VG.view(H, -1)  # (H, q1*q2)
    VV_T = VV @ VV.T  # (H, H)
    sum_G_squared = torch.sum(M_matrix * VV_T)  # scalar
    reg2 = lambd * sum_G_squared
    # ------------------------
    # Step 3: Positive terms
    # ------------------------
    # Pick correct index a_i^m from mat_ene_J
    # Z: (N1, M), we gather along dim=0
    # -> pos_J: (N1, M)
    q1, N1, M = mat_ene_J.shape
    Z_t = Z.T.unsqueeze(0)  # (1, M, N1)
    pos_J = mat_ene_J.permute(2,1,0).gather(
        dim=2, index=Z.T.unsqueeze(-1)
    ).squeeze(-1).permute(1,0)  # (N1, M)

    # Pick correct index a_i^m from mat_ene_G
    pos_G = mat_ene_G.permute(2,1,0).gather(
        dim=2, index=Z.T.unsqueeze(-1)
    ).squeeze(-1).permute(1,0)  # (N1, M)

    # Positive contribution: sum over i
    pos_term = pos_J + pos_G  # (N1, M)
    
    # ------------------------
    # Step 4: Log partition function
    # ------------------------
    # For each (i,m), compute:
    # logsumexp over a ∈ [1..q1]
    # exp( mat_ene_J[a,i,m] + mat_ene_G[a,i,m] )
    logits = mat_ene_J + mat_ene_G  # (q1, N1, M)
    logZ = torch.logsumexp(logits, dim=0)  # (N1, M)

    # ------------------------
    # Step 5: Final loss
    # ------------------------
    #loss_matrix =weights*(pos_term - logZ)  # (N1, M)
    loss_matrix= weights*torch.sum(pos_term-logZ,dim=0)
    loss_log=weights*torch.sum(logZ,dim=0)
    loss = -loss_matrix.mean()  # scalar
    print("model loss w/o reg", loss.item())
    loss_sum= weights*torch.sum(pos_term,dim=0)
    loss_log=weights*torch.sum(logZ,dim=0)
    loss_sum = -loss_sum.mean()  # scalar
    loss_log= loss_log.mean()
    print("model loss sum", loss_sum.item())
    print("model loss log", loss_log.item())
    print("reg1+reg2", (reg1+reg2).item())
    loss+=reg2+reg1
    del sf_J,sf_G, mat_ene_J,mat_ene_G, loss_matrix, VV_T
    torch.cuda.empty_cache()

    
    return loss

def loss_by_hand(J,J_PCA,Z,weights,lambd=0.001):
    loss_1=0
    loss_1_1=0
    loss_2=0
    L=J.shape[-1]
    q=J.shape[0]
    m=J_PCA.shape[-1]
    for seq in Z:
        for i in range(L):
            for j in range(L):
                if i==j:
                    continue
                loss_1+=J[seq[i],seq[j],i,j]
            for j in range(m):
                loss_1_1+=J_PCA[seq[i],seq[L+j],i,j]
            val=0
            for a in range(q):
                inside_exp=0
                for j in range(L):
                    if i==j:
                        continue
                    inside_exp+=J[a,seq[j],i,j]
                for j in range(m):
                    inside_exp+=J_PCA[seq[i],seq[L+j],i,j]
                val+=np.exp(inside_exp)
            loss_2-=np.log(val)
    print("by hand loss_1:",-loss_1)
    print("by hand loss_1_1:",-loss_1_1)
    print("by hand loss log:",-loss_2)
    loss=loss_1+loss_2+loss_1_1
    print("loss w/o reg: ", -loss)
    reg1= lambd*np.einsum("ijab,ijab",J,J)
    reg2= lambd*np.einsum("ijab,ijab",J_PCA,J_PCA)
    reg=reg2+reg1
    print("by hand reg:",reg)
    loss_tot=-loss+reg
    return loss_tot


In [24]:
H = 64
d= 10
N = 174
n_epochs = 500
nb_PCA_comp=2
loss_type = 'without_J'
family = 'jdoms' #'jdoms_bacteria_train2'
cwd = parent_dir
Q_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_PCA_2models_once_35_bins_fixed_masks/Q_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
K_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_PCA_2models_once_35_bins_fixed_masks/K_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
V_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_PCA_2models_once_35_bins_fixed_masks/V_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
H,d,N=Q_1.shape
q=V_1.shape[1]
model=AttentionModel(H,d,N,q,Q=Q_1,V=V_1,K=K_1)
torch.sum(model.Q-Q_1)
device = Q_1.device
L = Q_1.shape[-1]
W=attention_heads_from_model(model,Q_1,K_1,V_1)
print(W.shape)

i_indices = torch.arange(L, device=device).unsqueeze(1)
j_indices = torch.arange(L, device=device).unsqueeze(0)
mask = (i_indices != j_indices).float().unsqueeze(0)  # shape (1, L, L)
W = W * mask
    
# Compute Jtens
Jtens = torch.einsum('hri,hab->abri', W, V_1)  # Shape: (q, q, L, L)
q = Jtens.shape[0]
N = Jtens.shape[2]
print(q)
print(N)
print(Jtens.shape)
print(Jtens.shape[-1])
Q_1p = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_PCA_2models_once_35_bins_fixed_masks/Q_tensor_PCA.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
K_1p = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_PCA_2models_once_35_bins_fixed_masks/K_tensor_PCA.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
V_1p = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_PCA_2models_once_35_bins_fixed_masks/V_tensor_PCA.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
H,d,N1=Q_1p.shape
_,_,N2=K_1p.shape
_,q1,q2=V_1p.shape
model=AttentionModel_PCA(H,d,N1,N2,q1,q2,Q=Q_1p,V=V_1p,K=K_1p)
torch.sum(model.Q-Q_1p)
device = Q_1p.device
L = Q_1p.shape[-1]
W=attention_heads_from_model(model,Q_1p,K_1p,V_1p)
print(W.shape)

# i_indices = torch.arange(L, device=device).unsqueeze(1)
# j_indices = torch.arange(L, device=device).unsqueeze(0)
# mask = (i_indices != j_indices).float().unsqueeze(0)  # shape (1, L, L)
# W = W * mask
    
# Compute Jtens
Jtens_PCA = torch.einsum('hri,hab->abri', W, V_1p)  # Shape: (q, q, L, L)
q = Jtens.shape[0]
N = Jtens.shape[2]
print(Q_1p.shape)
print(K_1p.shape)
print(V_1p.shape)
print(q)
print(N)
print(Jtens_PCA.shape)

torch.Size([64, 63, 63])
21
63
torch.Size([21, 21, 63, 63])
63
torch.Size([32, 63, 2])
torch.Size([32, 10, 63])
torch.Size([32, 10, 2])
torch.Size([32, 21, 35])
21
63
torch.Size([21, 35, 63, 2])


In [31]:

def compute_mat_ene_cross_loop(Q, K, V, Z1, Z2, H1=0, H2=0, index_last_domain1=0):
    

    H, q1, q2 = V.shape
    N1, M = Z1.shape
    N2 = Z2.shape[0]

    sf = compute_attention_heads(Q, K, V, H1=H1, H2=H2, index_last_domain1=index_last_domain1)

    mat_ene = torch.zeros(q1, N1, M, device=device)

    for h in range(H):
        V_h = V[h]  # (q1, q2)

        # Expand indices
        Z1_exp = Z1[:, None, :].expand(N1, N2, M)  # (N1, N2, M)
        Z2_exp = Z2[None, :, :].expand(N1, N2, M)  # (N1, N2, M)

        # Flatten and index into V_h
        Z2_flat = Z2_exp.reshape(-1)
        V_selected_flat = V_h[:, Z2_flat]  # (q1, N1*N2*M)

        V_selected = V_selected_flat.view(q1, N1, N2, M)  # (q1, N1, N2, M)

        sf_h = sf[:, :, h]  # (N1, N2)

        mat_ene_h = torch.einsum('ij,qijm->qim', sf_h, V_selected)
        mat_ene += mat_ene_h

    return mat_ene, sf

# ----------------- Vectorized implementation -----------------
def compute_mat_ene_cross_vec(Q, K, V, Z1, Z2, H1=0, H2=0, index_last_domain1=0):
    

    H, q1, q2 = V.shape
    N1, M = Z1.shape
    N2 = Z2.shape[0]

    sf = compute_attention_heads(Q, K, V, H1=H1, H2=H2, index_last_domain1=index_last_domain1)

    # ---- Gather V across q2 dimension using Z2 ----
    Z2_exp = Z2.unsqueeze(0).unsqueeze(0)               # (1, 1, N2, M)
    Z2_exp = Z2_exp.expand(H, q1, N2, M)                # (H, q1, N2, M)

    V_exp = V.unsqueeze(2).unsqueeze(3).expand(H, q1, N2, M, q2)  # (H, q1, N2, M, q2)
    V_Z2 = torch.gather(V_exp, -1, Z2_exp.unsqueeze(-1))          # (H, q1, N2, M, 1)
    V_Z2 = V_Z2.squeeze(-1)                                       # (H, q1, N2, M)

    V_Z2 = V_Z2.permute(1, 2, 3, 0)   # (q1, N2, M, H)

    # Expand across N1 to align with sf
    V_Z2 = V_Z2.unsqueeze(1).expand(q1, N1, N2, M, H)  # (q1, N1, N2, M, H)

    # ---- Weighted sum ----
    mat_ene = torch.einsum('nkh,qnkmh->qnm', sf, V_Z2)  # (q1, N1, M)
    return mat_ene, sf




In [ ]:
from seq_utils import letters_to_nums, sequences_from_fasta, one_hot_seq_batch
family = 'jdoms_bacteria_train2'
file_test_data=r"C:\Users\youss\OneDrive\Bureau\master epfl\MA2\TP4 De los Rios\git_test\PLM-gen-DCA\Attention-DCA-main\CODE\DataAttentionDCA\jdoms\jdoms_bacteria_train2.fasta"

train_sequences = sequences_from_fasta(file_test_data)
train_sequences_num = [letters_to_nums(seq) for seq in train_sequences]
z_test=np.array(train_sequences_num[:1])
print(z_test.shape)
weights=np.array([1])
print(weights.shape)
Z, W = quickread(file_test_data,max_gap_frac=0.8)
Z=add_PCA_coords(Z.T,35).T
Z1=torch.from_numpy(Z[:-2,:])
Z2=torch.from_numpy(Z[-2:,:])


In [32]:
out_loop, sf_loop = compute_mat_ene_cross_loop(Q_1p, K_1p, V_1p, Z1,Z2)
out_vec, sf_vec = compute_mat_ene_cross_vec(Q_1p, K_1p, V_1p, Z1,Z2)


print("Max difference between loop and vectorized:", (out_loop - out_vec).abs().mean().item())

Max difference between loop and vectorized: 2.7052055884269066e-07


In [5]:
from seq_utils import letters_to_nums, sequences_from_fasta, one_hot_seq_batch
family = 'jdoms_bacteria_train2'
file_test_data=r"C:\Users\youss\OneDrive\Bureau\master epfl\MA2\TP4 De los Rios\git_test\PLM-gen-DCA\Attention-DCA-main\CODE\DataAttentionDCA\jdoms\jdoms_bacteria_train2.fasta"

train_sequences = sequences_from_fasta(file_test_data)
train_sequences_num = [letters_to_nums(seq) for seq in train_sequences]
z_test=np.array(train_sequences_num[:1])
print(z_test.shape)
weights=np.array([1])
print(weights.shape)
Z, W = quickread(file_test_data,max_gap_frac=0.8)
Z=add_PCA_coords(Z.T,35).T
Z1=Z[:-2,:1]
Z2=Z[-2:,:1]


(1, 63)
(1,)
Total sequences read: 14502
Sequences after filtering: 14502
Sampling 100000 pairs out of 105146751 total pairs.
Mean fraction of identical positions (sampled): 0.37202728570888266
Computed theta: 0.3268577458459699


100%|██████████| 14502/14502 [00:09<00:00, 1526.63it/s]


3265.70225762244


In [44]:
print(Jtens[1,0,1,0])
print(z_test)


tensor(0.0352, dtype=torch.float64)
[[11 12 19  3 17  9  5  7  8  3  5  0 15 13  2  3  7  8  8  0 19 14  3  9
   0  8  8 19  6 12  2 13 19  5  2 11 12  9  0  3  3  8 10 14  3  9 11  3
   0 19  2 19  9 15  8 11 15 20 20 20 20 20 20]]


In [8]:
print(loss_wo_J(Q_1,K_1,V_1,torch.from_numpy(z_test.T),torch.from_numpy(weights)))
print(loss_by_hand(Jtens,z_test,weights))


ale loss j :  -304.4594421386719
ale loss log:  364.5343017578125
V shape torch.Size([64, 21, 21])
64
ale loss:  60.074832916259766
ale reg:  6.265645503997803
66.3404769897461


C:\Users\youss\AppData\Local\Temp\ipykernel_22812\1871236379.py:238: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  val+=np.exp(inside_exp)
C:\Users\youss\AppData\Local\Temp\ipykernel_22812\1871236379.py:239: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  loss_2-=np.log(val)


by hand loss_: -304.4595947265625
by hand loss log: 364.53436279296875
by hand reg: 6.265655
66.34042358398438


In [33]:
z_test=np.concatenate((z_test,np.array([[17,5]])),axis=1)
print(z_test)

[[11 12 19  3 17  9  5  7  8  3  5  0 15 13  2  3  7  8  8  0 19 14  3  9
   0  8  8 19  6 12  2 13 19  5  2 11 12  9  0  3  3  8 10 14  3  9 11  3
   0 19  2 19  9 15  8 11 15 20 20 20 20 20 20 17  5]]


In [34]:
print(compute_loss(Q_1,K_1,V_1,torch.from_numpy(Z1),Q_1p,K_1p,V_1p,torch.from_numpy(Z1),torch.from_numpy(Z2),torch.from_numpy(weights)))
print(loss_by_hand(Jtens,Jtens_PCA,z_test,weights))

model loss w/o reg 63.527862548828125
model loss sum -279.27349853515625
model loss log 342.80133056640625
reg1+reg2 6.37666130065918
tensor(69.9045)


C:\Users\youss\AppData\Local\Temp\ipykernel_37236\1905655935.py:262: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  val+=np.exp(inside_exp)
C:\Users\youss\AppData\Local\Temp\ipykernel_37236\1905655935.py:263: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  loss_2-=np.log(val)


by hand loss_1: tensor(-279.2142)
by hand loss_1_1: tensor(-0.0591)
by hand loss log: tensor(342.8017)
loss w/o reg:  tensor(63.5284)
by hand reg: 6.3767147
tensor(69.9051)


Testing the energy calculation from the autoreg implementation 